In [ ]:
# ── Imports & paths ────────────────────────────────────────────────────────
import json
import re
from pathlib import Path
from collections import Counter

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT    = Path().resolve().parents[1]
RAW_DIR      = REPO_ROOT / "Data" / "Raw" / "non_western_fiction"
CLEAN_DIR    = REPO_ROOT / "Data" / "Clean"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ── Input files ────────────────────────────────────────────────────────────
# Use collected.json if notebook 02 has finished, otherwise api_results.json
API_PATH = RAW_DIR / "collected.json"
if not API_PATH.exists():
    API_PATH = RAW_DIR / "api_results.json"

LT_PATH    = RAW_DIR / "librarything_results.json"
OUTPUT_PATH = CLEAN_DIR / "cleaned_non_western_fiction.json"

print(f"✅ Setup complete")
print(f"   API source:  {API_PATH.name} — {'✅ found' if API_PATH.exists() else '❌ missing'}")
print(f"   LT source:   {LT_PATH.name} — {'✅ found' if LT_PATH.exists() else '❌ missing'}")
print(f"   Output:      {OUTPUT_PATH}")

✅ Setup complete
   API source:  collected.json — ✅ found
   LT source:   librarything_results.json — ✅ found
   Output:      C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Clean\cleaned_non_western_fantasy.json


In [2]:
# ── Load both datasets ─────────────────────────────────────────────────────
with open(API_PATH, encoding="utf-8") as f:
    api_books = json.load(f)

with open(LT_PATH, encoding="utf-8") as f:
    lt_books = json.load(f)

print(f"✅ Loaded datasets")
print(f"   API books:          {len(api_books):,}")
print(f"   LibraryThing books: {len(lt_books):,}")
print(f"   Total combined:     {len(api_books) + len(lt_books):,}")

# ── Quick field overview ───────────────────────────────────────────────────
print(f"\n── API book fields ───────────────────────────────────────────────")
print(list(api_books[0].keys()))

print(f"\n── LibraryThing book fields ──────────────────────────────────────")
print(list(lt_books[0].keys()))

✅ Loaded datasets
   API books:          4,713
   LibraryThing books: 13,849
   Total combined:     18,562

── API book fields ───────────────────────────────────────────────
['title', 'author_name', 'first_publish_year', 'subject', 'description', 'isbn', 'ol_key', 'language', 'source_query', 'needs_description', 'sg_genres', 'sg_moods', 'sg_pace', 'sg_rating', 'sg_url']

── LibraryThing book fields ──────────────────────────────────────
['title', 'work_id', 'lt_url', 'source_tag', 'source']


In [3]:
# ── Normalize to unified schema ────────────────────────────────────────────
# Every book gets the same fields regardless of source.
# Missing fields get sensible defaults.

def normalize_api_book(book):
    return {
        "title":               book.get("title", "").strip(),
        "author_name":         book.get("author_name", []),
        "first_publish_year":  book.get("first_publish_year", None),
        "description":         book.get("description", ""),
        "subject":             book.get("subject", []),
        "isbn":                book.get("isbn", []),
        "language":            book.get("language", []),
        "ol_key":              book.get("ol_key", ""),
        "lt_url":              "",
        "source_tag":          book.get("source_query", ""),
        "sg_genres":           book.get("sg_genres", []),
        "sg_moods":            book.get("sg_moods", []),
        "sg_pace":             book.get("sg_pace", ""),
        "sg_rating":           book.get("sg_rating", ""),
        "sg_url":              book.get("sg_url", ""),
        "needs_description":   book.get("needs_description", False),
        "source":              "open_library",
    }

def normalize_lt_book(book):
    return {
        "title":               book.get("title", "").strip(),
        "author_name":         [],
        "first_publish_year":  None,
        "description":         "",
        "subject":             [],
        "isbn":                [],
        "language":            [],
        "ol_key":              "",
        "lt_url":              book.get("lt_url", ""),
        "source_tag":          book.get("source_tag", ""),
        "sg_genres":           [],
        "sg_moods":            [],
        "sg_pace":             "",
        "sg_rating":           "",
        "sg_url":              "",
        "needs_description":   True,
        "source":              "librarything",
    }

# ── Apply normalization ────────────────────────────────────────────────────
api_normalized = [normalize_api_book(b) for b in api_books]
lt_normalized  = [normalize_lt_book(b)  for b in lt_books]

all_books = api_normalized + lt_normalized

print(f"✅ Normalization complete")
print(f"   API books:          {len(api_normalized):,}")
print(f"   LibraryThing books: {len(lt_normalized):,}")
print(f"   Total:              {len(all_books):,}")
print(f"\n── Unified fields ────────────────────────────────────────────────")
print(list(all_books[0].keys()))

✅ Normalization complete
   API books:          4,713
   LibraryThing books: 13,849
   Total:              18,562

── Unified fields ────────────────────────────────────────────────
['title', 'author_name', 'first_publish_year', 'description', 'subject', 'isbn', 'language', 'ol_key', 'lt_url', 'source_tag', 'sg_genres', 'sg_moods', 'sg_pace', 'sg_rating', 'sg_url', 'needs_description', 'source']


In [4]:
# ── Deduplication ──────────────────────────────────────────────────────────
# Priority: keep API book over LT book (more fields)
# Match on: normalized title + first author name

def normalize_title(title):
    """Lowercase, remove punctuation and extra spaces for matching."""
    title = title.lower().strip()
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', ' ', title)
    return title

seen = {}
deduplicated = []
duplicates_found = 0

# Process API books first so they take priority over LT books
for book in all_books:
    title_norm  = normalize_title(book["title"])
    author_norm = book["author_name"][0].lower().strip() if book["author_name"] else ""
    key         = f"{title_norm}||{author_norm}"

    # Fallback key — title only if no author
    if not author_norm:
        key = title_norm

    if not title_norm:
        continue  # skip empty titles

    if key not in seen:
        seen[key] = True
        deduplicated.append(book)
    else:
        duplicates_found += 1

print(f"✅ Deduplication complete")
print(f"   Before: {len(all_books):,}")
print(f"   After:  {len(deduplicated):,}")
print(f"   Removed {duplicates_found:,} duplicates")
print(f"\n   Source breakdown after dedup:")
sources = Counter(b["source"] for b in deduplicated)
for source, count in sources.most_common():
    print(f"     {source}: {count:,}")

✅ Deduplication complete
   Before: 18,562
   After:  18,080
   Removed 482 duplicates

   Source breakdown after dedup:
     librarything: 13,660
     open_library: 4,420


In [5]:
# ── Fixed subject matching — use exact or bounded matching ─────────────────
NON_FICTION_SUBJECT_KEYWORDS = [
    "cookery", "cooking (general)",
    "dictionaries", "language learning",
    "motion pictures -- production",
    "dvd", "blu-ray",
    "board games", "card games",
    "travel guides", "research"
]


NON_FICTION_TITLE_PATTERNS = [
    r'\bcookbook\b', r'\brecipes\b',
    r'\bdictionary\b', r'\bphrasebook\b',
    r'\bgrammar\b',
    r'\bscreenplay\b', r'\bdvd\b',
    r'\btravel guide\b', r'\bguidebook\b',
    r'\bsoundtrack\b',
]

BIOGRAPHY_SUBJECTS = ["biography", "autobiography"]

def is_non_fiction_media(book):
    title_lower = book["title"].lower()

    for pattern in NON_FICTION_TITLE_PATTERNS:
        if re.search(pattern, title_lower):
            return True

    subjects_lower = [s.lower() for s in book.get("subject", [])]

    for keyword in NON_FICTION_SUBJECT_KEYWORDS:
        if any(keyword in s for s in subjects_lower):
            return True

    # Biography — only match if subject IS exactly "biography" or "autobiography"
    # not if it appears inside a longer subject string
    for s in subjects_lower:
        for bio in BIOGRAPHY_SUBJECTS:
            if re.fullmatch(bio, s.strip()):
                return True

    return False

filtered_media = [b for b in deduplicated if not is_non_fiction_media(b)]
removed_media  = [b for b in deduplicated if is_non_fiction_media(b)]

print(f"✅ Media type filter complete")
print(f"   Before: {len(deduplicated):,}")
print(f"   After:  {len(filtered_media):,}")
print(f"   Removed: {len(removed_media):,}")
print(f"\n   Sample removed books:")
for b in removed_media[:25]:
    print(f"     {b['title']}")

✅ Media type filter complete
   Before: 18,080
   After:  17,656
   Removed: 424

   Sample removed books:
     A pure solar world
     Sun Ra's Chicago
     Recovering Black Storytelling in Qualitative Research
     The Visitor
     Of Solids and Surds
     Dictionary of modern Yoruba
     The first illustrated Yoruba dictionary
     English-Yoruba, Yoruba-English modern practical dictionary
     Yoruba-English/English-Yoruba concise dictionary
     A grammar of Yoruba
     There Was a Country
     Igbo grammar
     A short Igbo grammar
     Igbo-English dictionary
     Akan-English Dictionary
     Adinkra Alphabet Akan  Dictionary
     Akan terminology
     Shaka Zulu
     Compact Zulu dictionary
     Scholar's Zulu Dictionary
     Shaka
     Zulu dictionary
     Scholar's Zulu dictionary; English-Zulu, Zulu-English
     Scholar's Zulu dictionary
     Funda isi-Zulu


In [6]:
# ── Debug — why are good books being removed ──────────────────────────────
check_titles = ["Recovering Black Storytelling in Qualitative Research", "A pure solar world", "There Was a Country"]

for title in check_titles:
    matches = [b for b in removed_media if b["title"] == title]
    if matches:
        book = matches[0]
        print(f"\n── {book['title']} ──")
        print(f"   Subjects: {book['subject'][:5]}")
        # Check which pattern matched
        title_lower = book["title"].lower()
        for pattern in NON_FICTION_TITLE_PATTERNS:
            if re.search(pattern, title_lower):
                print(f"   Title pattern matched: {pattern}")
        subjects_lower = [s.lower() for s in book.get("subject", [])]
        for keyword in NON_FICTION_SUBJECT_KEYWORDS:
            for s in subjects_lower:
                if keyword in s:
                    print(f"   Subject matched: '{keyword}' in '{s}'")


── Recovering Black Storytelling in Qualitative Research ──
   Subjects: ['Qualitative research', 'Methodology', 'Storytelling in education', 'Afrofuturism', 'Feminist theory']
   Subject matched: 'research' in 'qualitative research'
   Subject matched: 'research' in 'research'
   Subject matched: 'research' in 'psychology / research & methodology'
   Subject matched: 'research' in 'social science / research'

── A pure solar world ──
   Subjects: ['Jazz', 'Jazz musicians', 'Biography', 'History and criticism', 'Jazz musicians, biography']

── There Was a Country ──
   Subjects: ['Nigerian Authors', 'Personal narratives', 'History', 'Biography', 'Nigeria, history, civil war, 1967-1970']


In [7]:
# ── noisy query filter ─────────────────────────────────────────────
NOISY_QUERIES = {
    "ubuntu", "maui", "cree", "pueblo", "mana"
}

FANTASY_SIGNALS = [
    "fantasy", "mythology", "folklore",
    "fairy tale", "legend", "magic", "supernatural"
]

LINUX_TITLE_SIGNALS = [
    "linux", "server", "ubuntu linux", "unleashed",
    "bible", "toolbox", "secrets", "administration",
    "guide to ubuntu", "ubuntu®", "ubuntu℗"
]

def is_noisy_query(book):
    source_tag  = book.get("source_tag", "").lower().strip()
    if source_tag not in NOISY_QUERIES:
        return False

    title_lower    = book.get("title", "").lower()
    subjects_lower = [s.lower() for s in book.get("subject", [])]

    # Always remove if Linux signals in title or subjects
    if any(signal in title_lower for signal in LINUX_TITLE_SIGNALS):
        return True
    if any("linux" in s or "operating system" in s for s in subjects_lower):
        return True

    # Keep only if fantasy/mythology specifically
    if any(
        any(signal in s for signal in FANTASY_SIGNALS)
        for s in subjects_lower
    ):
        return False

    return True

filtered_noisy = [b for b in filtered_media if not is_noisy_query(b)]
removed_noisy  = [b for b in filtered_media if is_noisy_query(b)]

print(f"✅ Noisy query filter complete")
print(f"   Before: {len(filtered_media):,}")
print(f"   After:  {len(filtered_noisy):,}")
print(f"   Removed: {len(removed_noisy):,}")

print(f"\n   Sample kept Ubuntu books:")
for b in [b for b in filtered_noisy if b.get("source_tag") == "Ubuntu"][:8]:
    print(f"     {b['title']} — {b['subject'][:2]}")

print(f"\n   Sample kept Maui books:")
for b in [b for b in filtered_noisy if b.get("source_tag") == "Maui"][:8]:
    print(f"     {b['title']} — {b['subject'][:2]}")

✅ Noisy query filter complete
   Before: 17,656
   After:  15,745
   Removed: 1,911

   Sample kept Ubuntu books:
     Ubuntu — ['Fiction, fantasy, general', 'Philosophy']

   Sample kept Maui books:


In [8]:
# ── Check all noisy query results ──────────────────────────────────────────
for query in ["Cree", "Pueblo", "Mana"]:
    kept = [b for b in filtered_noisy if b.get("source_tag") == query]
    print(f"\n{query} books kept: {len(kept)}")
    for b in kept[:5]:
        print(f"  {b['title']} — {b['subject'][:2]}")


Cree books kept: 3
  Cree — ['Cree Indians', 'Folklore']
  Cree legends — ['Juvenile literature', 'Cree Indians']
  Cree legends — ['Folklore', 'Cree Indians']

Pueblo books kept: 5
  Wolf Brother — ['Bears', 'Ficción']
  Arrow to the sun — ['Folklore', 'Pueblo Indians']
  The peoples of Middle-Earth — ['English Fantasy literature', 'Middle Earth (Imaginary place)']
  Ghost Town at Sundown — ['Tree houses', 'Frontier and pioneer life']
  Pueblo stories & storytellers — ['Storytellers in art', 'Pueblo pottery']

Mana books kept: 0


In [9]:
# ── Updated licensed/wrong genre filter ───────────────────────────────────
LICENSED_TITLE_SIGNALS = [
    "star wars", "marvel comics", "black panther comics",
    "avengers", "spider-man", "moana", "lilo and stitch", "disney",
]

# Mainstream manga and western comics blocklist
MAINSTREAM_MANGA = [
    "naruto", "bleach", "dragon ball", "dragonball", "one piece",
    "death note", "yu-gi-oh", "sailor moon", "inuyasha", "pokemon",
    "fullmetal alchemist", "attack on titan", "my hero academia",
    "sword art online", "fairy tail", "gto", "negima", "neon genesis",
    "evangelion", "akira", "ghost in the shell",
]

WESTERN_COMICS = [
    "punisher", "wolverine", "batman", "superman", "x-men",
    "wonder woman", "justice league", "green lantern",
]

WRONG_GENRE_SUBJECTS = [
    "ben-hur", "ben hur",
    "romance fiction",
    "true crime",
]

def is_licensed_or_wrong_genre(book):
    title_lower    = book.get("title", "").lower()
    subjects_lower = [s.lower() for s in book.get("subject", [])]
    source_tag     = book.get("source_tag", "").lower()

    # Always drop western comics
    if any(signal in title_lower for signal in WESTERN_COMICS):
        return True

    # Always drop mainstream manga
    if any(signal in title_lower for signal in MAINSTREAM_MANGA):
        return True

    # Always drop licensed titles
    if any(signal in title_lower for signal in LICENSED_TITLE_SIGNALS):
        return True

    # Drop manga/comics UNLESS they are African or non-western tagged
    non_western_comic_tags = [
        "african fantasy", "indigenous futurisms",
        "southeast asian fantasy", "south asian fantasy",
        "middle eastern fantasy", "latin american fantasy"
    ]
    subjects_and_tags = subjects_lower + [source_tag]
    is_comic = any(
        s in ["manga", "comics (graphic works)", "graphic novels"]
        for s in subjects_lower
    )
    if is_comic:
        if any(tag in source_tag for tag in non_western_comic_tags):
            return False  # keep non-western comics
        return True  # drop everything else

    # Wrong genre subjects
    if any(
        any(signal in s for signal in WRONG_GENRE_SUBJECTS)
        for s in subjects_lower
    ):
        return True

    return False

filtered_licensed = [b for b in filtered_noisy if not is_licensed_or_wrong_genre(b)]
removed_licensed  = [b for b in filtered_noisy if is_licensed_or_wrong_genre(b)]

print(f"✅ Licensed/wrong genre filter complete")
print(f"   Before: {len(filtered_noisy):,}")
print(f"   After:  {len(filtered_licensed):,}")
print(f"   Removed: {len(removed_licensed):,}")

# Sanity check
kept_african_comics = [b for b in filtered_licensed 
                       if "malika" in b["title"].lower() 
                       or "djeliya" in b["title"].lower()]
print(f"\n   African comics kept: {len(kept_african_comics)}")
for b in kept_african_comics:
    print(f"     {b['title']}")

print(f"\n   Sample removed:")
for b in removed_licensed[:15]:
    print(f"     {b['title']}")

✅ Licensed/wrong genre filter complete
   Before: 15,745
   After:  15,532
   Removed: 213

   African comics kept: 2
     Djeliya
     Malika - Warrior Queen Part Two

   Sample removed:
     Holiday Fantasy
     The punisher wolverine
     Island for Two
     Bleach
     Naruto 5
     Dragonball (Doragon bo-ru)
     Battle royale
     Dragon Ball Z; Thank You
     Mahō sensei Negima
     Yu-Gi-Oh!
     Sailor Moon, Vol. 1
     Death Note 1
     Naruto, Vol. 1
     GTO
     The Lost Warrior


In [10]:
BLOCKED_AUTHORS = [
    "j. k. rowling", "j.k. rowling", "joanne rowling",
    "sarah j. maas", "sarah maas",
    "australian bureau of statistics",
    "william shakespeare",
]

def has_blocked_author(book):
    authors_lower = [a.lower().strip() for a in book.get("author_name", [])]
    # Only block if it's the sole/primary author (first in list)
    primary_author = authors_lower[0] if authors_lower else ""
    return any(blocked in primary_author for blocked in BLOCKED_AUTHORS)

filtered_authors = [b for b in filtered_licensed if not has_blocked_author(b)]
removed_authors  = [b for b in filtered_licensed if has_blocked_author(b)]

print(f"✅ Author filter complete")
print(f"   Before: {len(filtered_licensed):,}")
print(f"   After:  {len(filtered_authors):,}")
print(f"   Removed: {len(removed_authors):,}")
for b in removed_authors[:10]:
    print(f"     {b['title']} — {b['author_name'][0]}")

✅ Author filter complete
   Before: 15,532
   After:  15,493
   Removed: 39
     A Court of Thorns and Roses — Sarah J. Maas
     Throne of Glass — Sarah J. Maas
     Heir of Fire — Sarah J. Maas
     Empire of Storms — Sarah J. Maas
     A Court of Wings and Ruin — Sarah J. Maas
     A Court of Frost and Starlight — Sarah J. Maas
     Tower of Dawn — Sarah J. Maas
     Queen of Shadows — Sarah J. Maas
     Kingdom of Ash — Sarah J. Maas
     Harry Potter and the Deathly Hallows — J. K. Rowling


In [11]:
# ── Debug Shakespeare ──────────────────────────────────────────────────────
shakespeare_books = [b for b in filtered_licensed 
                     if any("shakespeare" in a.lower() 
                     for a in b.get("author_name", []))]
print(f"Shakespeare books: {len(shakespeare_books)}")
for b in shakespeare_books[:5]:
    print(f"  {b['title']} — {b['author_name']}")

Shakespeare books: 18
  One-Hundred-and-One Read-Aloud Classics — ['Pamela Horn', 'Judy Blume', 'Beverly Cleary', 'Astrid Lindgren', 'Laura Ingalls Wilder', 'Jack London', 'Wilson Rawls', 'John Steinbeck', 'Lewis Carroll', 'Roald Dahl', 'Charles Dickens', 'C. S. Lewis', 'William Shakespeare', 'J.R.R. Tolkien', 'E. B. White']
  The Tragedy of Titus Andronicus — ['William Shakespeare']
  Love's Labour's Lost — ['William Shakespeare']
  Twelfth night — ['William Shakespeare']
  Titus Andronicus — ['William Shakespeare']


In [12]:
# ── Count unique authors in cleaned dataset ────────────────────────────────
from collections import Counter

all_authors = []
for book in filtered_authors:
    for author in book.get("author_name", []):
        if author.strip():
            all_authors.append(author.strip())

author_counts = Counter(all_authors)
print(f"Total author mentions: {len(all_authors):,}")
print(f"Unique authors:        {len(author_counts):,}")
print(f"\nTop 10 most prolific:")
for author, count in author_counts.most_common(10):
    print(f"  {count:2d} books — {author}")

Total author mentions: 5,769
Unique authors:        4,494

Top 10 most prolific:
  30 books — Nnedi Okorafor
  30 books — Salman Rushdie
  25 books — Yoon Ha Lee
  22 books — Roshani Chokshi
  22 books — Ken Liu
  21 books — Silvia Moreno-Garcia
  20 books — Ambelin Kwaymullina
  19 books — A W. Reed
  18 books — Zoraida Córdova
  17 books — Aliette de Bodard


In [13]:
# ── Fiction/Fantasy filter ─────────────────────────────────────────────────
FICTION_SIGNALS = [
    "fantasy", "fiction", "speculative fiction",
    "science fiction", "magical realism",
    "folklore", "mythology retelling"
]

FANTASY_SOURCE_TAGS = [
    "Afrofuturism", "Africanfuturism", "Africanjujuism",
    "Wuxia", "Xianxia", "Xuanhuan",
    "Yokai", "Kitsune", "Oni", "Tengu", "Shinto",
    "Djinn", "Ifrit", "Marid",
    "Orisha", "Vodou", "Anansi",
    "Aswang", "Diwata", "Pontianak",
    "Rakshasa", "Naga", "Asura",
    "Quetzalcoatl", "Xibalba",
    "Skinwalker", "Wendigo", "Thunderbird",
    "Dreamtime", "Bunyip",
    "Gumiho", "Dokkaebi",
]

def is_fantasy_fiction(book):
    source     = book.get("source", "")
    source_tag = book.get("source_tag", "")
    subjects   = book.get("subject", [])

    # OL books — filter by subjects
    if source == "open_library":
        subjects_lower = " ".join(subjects).lower()
        return any(sig in subjects_lower for sig in FICTION_SIGNALS)

    # LT books — filter by source tag
    if source == "librarything":
        return source_tag in FANTASY_SOURCE_TAGS

    return False

fantasy_filtered = [b for b in filtered_authors if is_fantasy_fiction(b)]
removed_non_fantasy = len(filtered_authors) - len(fantasy_filtered)

print(f"✅ Fiction/fantasy filter complete")
print(f"   Before: {len(filtered_authors):,}")
print(f"   After:  {len(fantasy_filtered):,}")
print(f"   Removed: {removed_non_fantasy:,}")
print(f"\n   Source breakdown:")
from collections import Counter
for source, count in Counter(b['source'] for b in fantasy_filtered).most_common():
    print(f"     {source}: {count:,}")

✅ Fiction/fantasy filter complete
   Before: 15,493
   After:  7,742
   Removed: 7,751

   Source breakdown:
     librarything: 6,078
     open_library: 1,664


In [19]:
# ── Check how many would remain after filtering study/poetry ──────────────
STUDY_POETRY_SUBJECTS = [
    "poetry", "poems", "verse",
    "study", "criticism", "history and criticism",
    "academic", "scholarly",
    "textbook", "handbook",
    "essays", "lectures",
    "bibliography",
]

STUDY_POETRY_TITLE_SIGNALS = [
    "study", "critical", "companion to",
    "introduction to", "guide to",
    "handbook", "anthology of criticism",
    "essays on", "lectures on",
]

def is_study_or_poetry(book):
    subjects_lower = " ".join(book.get("subject", [])).lower()
    title_lower    = book.get("title", "").lower()

    if any(sig in subjects_lower for sig in STUDY_POETRY_SUBJECTS):
        return True
    if any(sig in title_lower for sig in STUDY_POETRY_TITLE_SIGNALS):
        return True
    return False

would_remove = [b for b in fantasy_filtered if is_study_or_poetry(b)]
would_keep   = [b for b in fantasy_filtered if not is_study_or_poetry(b)]

print(f"Current total:        {len(fantasy_filtered):,}")
print(f"Would remove:         {len(would_remove):,}")
print(f"Would keep:           {len(would_keep):,}")
print(f"\nSample removed:")
for b in would_remove[:10]:
    print(f"  {b['title']} — {b.get('subject', [])[:2]}")

Current total:        7,742
Would remove:         262
Would keep:           7,480

Sample removed:
  Afrofuturism Rising — ['Afrofuturism', 'American literature']
  Janelle Monáe's Queer Afrofuturism — ['Criticism and interpretation', 'Popular music']
  Afro-Future Females — ['American Science fiction', 'Women authors']
  Afrofuturism: The World of Black Sci-Fi and Fantasy Culture — ['Race identity', 'Social aspects']
  Off the planet — ['Motion pictures', 'Motion picture music']
  We Travel the Space Ways — ['Civilization, history', 'Afrofuturism']
  Dark Matter — ['Criticism', 'American Science fiction']
  How Long 'Til Black Future Month — ['Fiction, science fiction, short stories', 'New York Times reviewed']
  Victor Brauner — ["Abbaye Sainte-Croix (Les Sables-d'Olonne, France)", 'Catalogs']
  Distrust that particular flavor — ['Popular culture', 'Canadian essays']


In [20]:
def is_study_or_poetry(book):
    subjects_lower = " ".join(book.get("subject", [])).lower()
    title_lower    = book.get("title", "").lower()

    # Only remove if CLEARLY non-fiction academic/poetry
    if any(sig in subjects_lower for sig in [
        "poetry", "poems", "verse",
        "history and criticism",
        "handbooks, manuals",
        "catalogs",
        "bibliography",
        "lectures",
    ]):
        return True

    # Only remove on very specific title signals
    if any(sig in title_lower for sig in [
        "companion to", "introduction to",
        "handbook of", "guide to",
    ]):
        return True

    return False

would_remove = [b for b in fantasy_filtered if is_study_or_poetry(b)]
would_keep   = [b for b in fantasy_filtered if not is_study_or_poetry(b)]

print(f"Current total:  {len(fantasy_filtered):,}")
print(f"Would remove:   {len(would_remove):,}")
print(f"Would keep:     {len(would_keep):,}")
print(f"\nSample removed:")
for b in would_remove[:10]:
    print(f"  {b['title']} — {b.get('subject', [])[:2]}")

Current total:  7,742
Would remove:   199
Would keep:     7,543

Sample removed:
  Afrofuturism Rising — ['Afrofuturism', 'American literature']
  Janelle Monáe's Queer Afrofuturism — ['Criticism and interpretation', 'Popular music']
  Afro-Future Females — ['American Science fiction', 'Women authors']
  Afrofuturism: The World of Black Sci-Fi and Fantasy Culture — ['Race identity', 'Social aspects']
  Off the planet — ['Motion pictures', 'Motion picture music']
  We Travel the Space Ways — ['Civilization, history', 'Afrofuturism']
  Victor Brauner — ["Abbaye Sainte-Croix (Les Sables-d'Olonne, France)", 'Catalogs']
  Demonic vision — ['African Americans in literature', 'American fiction']
  Black Atlantic speculative fictions — ['African Americans in literature', 'African American authors']
  Speculative Blackness — ['American fiction, african american authors, history and criticism', 'Science fiction, history and criticism']


In [21]:
final_books = [b for b in fantasy_filtered if not is_study_or_poetry(b)]

print(f"✅ Study/poetry filter complete")
print(f"   Before: {len(fantasy_filtered):,}")
print(f"   After:  {len(final_books):,}")
print(f"   Removed: {len(fantasy_filtered) - len(final_books):,}")

✅ Study/poetry filter complete
   Before: 7,742
   After:  7,543
   Removed: 199


In [24]:
NONFICTION_SIGNALS = [
    "nonfiction", "non-fiction", "non fiction",
    "juvenile nonfiction", "adult nonfiction",
    "anthropology", "social evolution",
    "description and travel", "pictorial works",
    "christianity and culture", "religious aspects of ethnicity",
]

def is_nonfiction(book):
    subjects_lower = " ".join(book.get("subject", [])).lower()
    return any(sig in subjects_lower for sig in NONFICTION_SIGNALS)

would_remove = [b for b in final_books if is_nonfiction(b)]
would_keep   = [b for b in final_books if not is_nonfiction(b)]

print(f"Current total:  {len(final_books):,}")
print(f"Would remove:   {len(would_remove):,}")
print(f"Would keep:     {len(would_keep):,}")
print(f"\nSample removed:")
for b in would_remove[:10]:
    print(f"  {b['title']} — {b.get('subject', [])[:2]}")

Current total:  7,543
Would remove:   57
Would keep:     7,486

Sample removed:
  Corduroy — ['Chinese language materials', 'Corduroy (Fictitions character)']
  Politics and Post-colonial Theory — ['Literary Criticism', 'Nonfiction']
  Helena's Voyage — ['Voyages and travels', 'Judaism']
  2312 — ['FICTION / Science Fiction / High Tech', 'FICTION / Science Fiction / General']
  Handa's Surprise — ['Amistad', 'Arabic language materials']
  Voodoos and obeahs — ['Blacks', 'Voodooism']
  The Haitian vodou handbook — ['Religion', 'Nonfiction']
  Anansi and the tug of war — ['Legends', 'Juvenile fiction']
  Mami Wata — ['Child and youth non-fiction', "Children's fiction"]
  African myth of creation in African form of writing — ['Christianity and culture', 'Religious life and customs']


In [25]:
# ── Debug specific books ───────────────────────────────────────────────────
check = ["2312", "Anansi and the tug of war", "Mami Wata"]
for title in check:
    matches = [b for b in final_books if b["title"] == title]
    if matches:
        book = matches[0]
        subjects_lower = " ".join(book.get("subject", [])).lower()
        print(f"\n{title}")
        print(f"  Subjects: {book['subject'][:5]}")
        for sig in NONFICTION_SIGNALS:
            if sig in subjects_lower:
                print(f"  Triggered by: '{sig}'")


2312
  Subjects: ['FICTION / Science Fiction / High Tech', 'FICTION / Science Fiction / General', 'FICTION / Science Fiction / Space Opera', 'Space colonies', 'Fiction']
  Triggered by: 'anthropology'

Anansi and the tug of war
  Subjects: ['Legends', 'Juvenile fiction', 'Spiders', "Children's stories", 'Anansi (Legendary character)']
  Triggered by: 'pictorial works'

Mami Wata
  Subjects: ['Child and youth non-fiction', "Children's fiction", 'Girls, fiction']
  Triggered by: 'non-fiction'


In [26]:
NONFICTION_SIGNALS = [
    "nonfiction",
    "non-fiction", 
    "juvenile nonfiction",
    "adult nonfiction",
    "child and youth non-fiction",
]

def is_nonfiction(book):
    subjects_lower = [s.lower() for s in book.get("subject", [])]
    return any(
        any(sig in s for sig in NONFICTION_SIGNALS)
        for s in subjects_lower
    )

would_remove = [b for b in final_books if is_nonfiction(b)]
would_keep   = [b for b in final_books if not is_nonfiction(b)]

print(f"Current total:  {len(final_books):,}")
print(f"Would remove:   {len(would_remove):,}")
print(f"Would keep:     {len(would_keep):,}")
print(f"\nSample removed:")
for b in would_remove[:10]:
    print(f"  {b['title']} — {b.get('subject', [])[:2]}")

Current total:  7,543
Would remove:   37
Would keep:     7,506

Sample removed:
  Politics and Post-colonial Theory — ['Literary Criticism', 'Nonfiction']
  The Haitian vodou handbook — ['Religion', 'Nonfiction']
  Mami Wata — ['Child and youth non-fiction', "Children's fiction"]
  Religion Explained — ['Nonfiction', 'Philosophy']
  Color, culture & creed — ['Faith', 'Religious aspects of Ethnicity']
  Our Kind — ['Anthropology', 'Social evolution']
  The spirit of Asia — ['Description and travel', 'Lugar sagrado']
  Broken Spirits — ['Medical', 'Nonfiction']
  Move of the Holy Spirit in the 10/40 Window — ['Juvenile Nonfiction', 'Religion - Christianity']
  Shinto, a short history — ['History', 'Nonfiction']


In [27]:
final_books = [b for b in final_books if not is_nonfiction(b)]

print(f"✅ Nonfiction filter complete")
print(f"   After:   {len(final_books):,}")

✅ Nonfiction filter complete
   After:   7,506


In [28]:
# ── Save final cleaned dataset ─────────────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_books, f, ensure_ascii=False, indent=2)

from collections import Counter
source_counts = Counter(b["source"] for b in final_books)
has_desc      = sum(1 for b in final_books if b.get("description"))
has_author    = sum(1 for b in final_books if b.get("author_name"))
has_year      = sum(1 for b in final_books if b.get("first_publish_year"))
has_cover     = sum(1 for b in final_books if b.get("cover_url"))

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  Final cleaning summary")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Total books:          {len(final_books):,}")
print(f"\n  Source breakdown:")
for source, count in source_counts.most_common():
    print(f"    {source}: {count:,}")
print(f"\n  Field coverage:")
print(f"    Has description:  {has_desc:,} ({has_desc/len(final_books)*100:.1f}%)")
print(f"    Has author:       {has_author:,} ({has_author/len(final_books)*100:.1f}%)")
print(f"    Has publish year: {has_year:,} ({has_year/len(final_books)*100:.1f}%)")
print(f"    Has cover URL:    {has_cover:,} ({has_cover/len(final_books)*100:.1f}%)")
print(f"\n  Saved to: {OUTPUT_PATH}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Final cleaning summary
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total books:          7,506

  Source breakdown:
    librarything: 5,990
    open_library: 1,516

  Field coverage:
    Has description:  1,328 (17.7%)
    Has author:       1,504 (20.0%)
    Has publish year: 1,514 (20.2%)
    Has cover URL:    0 (0.0%)

  Saved to: C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Clean\cleaned_non_western_fantasy.json
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [15]:
# ── Check what graphic novels/manga we have ────────────────────────────────
manga_removed = [b for b in removed_licensed 
                 if any(s in str(b.get("subject", [])).lower() 
                 for s in ["manga", "graphic novel", "comics"])]

print(f"Removed via manga/comics subjects: {len(manga_removed)}")
for b in manga_removed[:20]:
    print(f"  {b['title']} — {b['source_tag']}")

Removed via manga/comics subjects: 101
  The punisher wolverine — African fantasy
  Bleach — Asian fantasy
  Naruto 5 — Asian fantasy
  Dragonball (Doragon bo-ru) — Asian fantasy
  Battle royale — Asian fantasy
  Dragon Ball Z; Thank You — Asian fantasy
  Mahō sensei Negima — Asian fantasy
  Yu-Gi-Oh! — Asian fantasy
  Sailor Moon, Vol. 1 — Asian fantasy
  Death Note 1 — Asian fantasy
  Naruto, Vol. 1 — Asian fantasy
  GTO — Asian fantasy
  The Lost Warrior — Asian fantasy
  Dragon Ball Z (Dragon Ball Z — Asian fantasy
  InuYasha, Vol. 19 — Asian fantasy
  Pokemon Adventures — Asian fantasy
  Naruto Uzumaki Naruto — Asian fantasy
  鋼の錬金術師 1 — Asian fantasy
  Kareshi Kanojo no jijou — Asian fantasy
  Death Note, Vol. 2 — Asian fantasy


In [16]:
# ── Debug Ghost Town at Sundown ────────────────────────────────────────────
ghost = [b for b in filtered_noisy if b["title"] == "Ghost Town at Sundown"]
if ghost:
    print(f"Subjects: {ghost[0]['subject']}")
    print(f"Source tag: {ghost[0]['source_tag']}")

Subjects: ['Tree houses', 'Frontier and pioneer life', 'Spanish language materials', 'Fiction', 'Juvenile fiction', 'Magic', 'Time travel', 'West (U.S.)', 'History', 'Western stories', 'Magia', 'Frontera y exploradores', 'Vida', 'Casas en árboles', 'Ficción juvenil', 'Viajes a través del tiempo', "Children's fiction", 'Tree houses, fiction', 'West (u.s.), fiction', 'Frontier and pioneer life, fiction', 'Time travel, fiction', 'Magic, fiction', 'Fantasy fiction', 'Kinderbuch ab 8 Jahren', 'Mädchen', 'Junge', 'Schwester', 'Bruder', 'Geschwister', 'Baumhaus', 'versteckt', 'verborgen', 'Zeitreise', 'Abenteuer', 'Geisterstadt', 'Wilder Westen', 'Steppe', 'Berge', 'Pferde', 'Cowboys', 'Mustangs']
Source tag: Pueblo


In [17]:
# ── Debug — check what Ubuntu books we're keeping vs removing ──────────────
ubuntu_removed = [b for b in removed_noisy if b["source_tag"] == "Ubuntu"]
ubuntu_kept    = [b for b in filtered_noisy if b.get("source_tag") == "Ubuntu"]

print(f"Ubuntu books removed: {len(ubuntu_removed)}")
print(f"Ubuntu books kept:    {len(ubuntu_kept)}")

print(f"\nSample kept Ubuntu books:")
for b in ubuntu_kept[:10]:
    print(f"  {b['title']} — subjects: {b['subject'][:3]}")

print(f"\nSample removed Ubuntu books (last 10):")
for b in ubuntu_removed[-10:]:
    print(f"  {b['title']} — subjects: {b['subject'][:3]}")

Ubuntu books removed: 51
Ubuntu books kept:    1

Sample kept Ubuntu books:
  Ubuntu — subjects: ['Fiction, fantasy, general', 'Philosophy']

Sample removed Ubuntu books (last 10):
  UBUNTU contributionism — subjects: ['Social aspects', 'Money', 'Banks and banking']
  Sundowner Ubuntu — subjects: ['Private investigators', 'Russell Quant (Fictitious character)', 'Gay men']
  Ludic Ubuntu Ethics — subjects: ['Philosophy']
  Hoops Africa — subjects: ['Basketball', 'Basketball players']
  Ubuntu — subjects: ['Operating systems (Computers)', 'UNIX', 'System Administration']
  Ubuntu and Western Monotheism — subjects: ['Philosophy', 'Ubuntu (Philosophy)', 'Monotheism']
  Practical Guide to Ubuntu Linux — subjects: ['Linux (computer operating system)', 'Operating systems (computers)', 'Operating systems (Computers)']
  Ubuntu and Personhood — subjects: ['Identity (philosophical concept)', 'Philosophy, african', 'Social groups']
  Family BondZone! — subjects: ['Family', 'Games']
  The official